<img src="images/logodwengo.png" alt="LogoDwengo" width="150"/>

<div>
    <font color=#690027 markdown="1">
<h1>SIMULATE AN EPIDEMIC: A DISEASE OUTBREAK IN A SOCIAL NETWORK</h1>    </font>
</div>

<div class="alert alert-box alert-success">
In this project, you study how diseases can spread through a (social) network. You investigate how the structure of a network can influence how quickly a disease is passed on. Finally, you will also look at various strategies to combat the spread of a disease.<br>In this notebook, you apply the SIR model within a social network.</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.spatial import distance_matrix

## A disease outbreak in a social network

Take a look now at how you can translate the SIR-disease diffusion model to the language of networks. <br>Using a general network, you will set up a much more realistic model. No more continuous approximation! This model surprisingly aligns better with reality, and moreover, it is also much simpler to grasp and simulate. You can obtain an exact solution without needing derivatives or other advanced mathematical techniques!
### Disease dynamics on a network
Instead of keeping track of the number of $S$-, $I$- and $R$-individuals over time as in the standard SIR model, you will monitor the state of each node in the network. Time will not vary continuously but will now pass in discrete steps: $t = 0, 1, 2, 3, \ldots$. <br>- The state of node number $i$ at time $t$ is described by $N_i^t\in \{S, I, R\}$. This means that at time $t$, node $i$ can be in state $S$ (susceptible), $I$ (infected), or $R$ (resistant).- The change in state of the nodes is described based on a few simple rules. Analogous to the original SIR model that has two parameters, beta (the infection rate) and gamma (the recovery rate), the SIR model for a network also has two parameters.

#### Susceptible and infected people
You first limit yourself to susceptible and infected individuals. You assume that susceptible individuals can become infected, and infected individuals can become resistant. So there is no possible transition from infected to susceptible and neither from susceptible to resistant. Consider the following rules:
- If a node is in state $S$ at time $t$, then each **infected** neighbor has a chance $p_\text{inf}$ of transmitting the disease. The node goes to state $I$ if at least one neighbor transmits the disease.- If a node is in state $I$ at time $t$, then it moves to state $R$ with a probability $p_\text{res}$.

So, suppose a node is in state $S$, and it has $k$ neighbors who are in state $I$. The probability that no neighbor passes on the disease, is then:
I'm sorry, there is no text in your prompt for me to translate. Please provide the text in the Dutch language.(1-p_\text{inf})^k,The input doesn't contain any Dutch text to translate into English.
so the chance that the disease is transmitted, and thus a transition from state $S$ to $I$ occurs, is:
The given input is blank, and so the output remains blank.1 - (1-p_\text{inf})^k\,.Since the input provided does not contain any text to be translated, the output will be the same as the input.
You used the product rule and the complement rule from probability theory here.


#### ExampleConsider the node outlined in blue in the figure below. Suppose that $p_\text{inf}=0.2$, what is the probability that one of the three sick neighbors passes on the disease?
![](images/diseasespread.png)<center> Figure 1.</center>
You calculate this with the following code:

In [ ]:
p_inf = 0.2
k = 3

p_ziekte_doorgegeven = 1 - (1 - p_inf)**k

print("Kans om de ziekte te krijgen is:", p_ziekte_doorgegeven)

The effective transmission of the disease can be simulated with NumPy, where `np.random.rand()` generates a random number, uniformly distributed between 0 and 1. <br> You do that with the code in the following code cell. Run that cell several times for the simulation.

In [ ]:
# voorbeeld
p_ziekte_doorgegeven > np.random.rand()

In [ ]:
# voorbeeld
p_ziekte_doorgegeven > np.random.rand()

In [ ]:
# voorbeeld
p_ziekte_doorgegeven > np.random.rand()

With `True`, the disease is effectively transmitted, with `False` it is not. Note that a random factor has been built into the simulation.

> **Exercise 1**: Assume that $p_\text{inf}=1$ (everyone who is ill immediately passes on the disease to all his or her neighbors in the network). Initially, only nodes 1 and 11 are infected in the example network from Figure 3 of the previous notebook on social networks.<br>- Who all are infected in the next step?- And in the next step?

Answer:

### ImplementationYou can easily implement the model in Python using SciPy. <br>First, you will generate a simple social network to illustrate this model:- You generate a population of `n` people for this. To keep it visual, these are represented as points in the $x,y$-plane.- Afterwards, you generate a connection matrix that indicates whether there is a connection between the nodes.

#### First you generate the nodes of the network. At the same time, you generate the distance between the nodes.

In [ ]:
def genereer_populatie(n):
    """Genereren van punten en bepalen van hun onderlinge afstand."""
    # n punten genereren, uniform in het xy-vlak
    X = np.random.rand(n, 2)
    # alle paarsgewijze afstanden tussen n punten
    D = distance_matrix(X, X)
    return X, D

In [ ]:
# populatie van netwerk van 200 punten genereren
n = 200
X, D = genereer_populatie(n)

In [ ]:
print(X,D)

The distances between two individuals together form the distance matrix $D$.

In [ ]:
# X bestaat uit 200 koppels en D is 200x200-matrix
print(X.shape, D.shape)

#### Now you generate the connection matrix V.

To obtain a simple model for the connection matrix V, you assume that the probability that $v_{ij}=1$, that is, that nodes $i$ and $j$ are connected, is given by:
The input provided doesn't contain any Dutch text to translate. Please provide the correct input.p_{ij} = \exp(-\alpha \, d_{ij})\,.The input provided does not contain any text to translate. It is necessary to provide a text in Dutch language, or in html, markdown or python code with Dutch comments, so that it can be translated into English.
**Here it applies that the chance of a connection between nodes $i$ and $j$ decreases as the distance between the two nodes increases.** <br>$\alpha$ is a parameter ($\alpha \geq 0$) that controls this correlation. A large value of $\alpha$ ensures that two widely separated nodes have a very small chance of being connected. For a small value of $\alpha$, this is still possible. Moreover, it holds that the greater the distance between two nodes, the smaller the chance of a connection.

In [ ]:
# illustratie van effect van waarde van alpha
plt.figure() 

xwaarden = np.linspace(0, 10, 100)
plt.plot(xwaarden, np.exp(-0.1 * xwaarden), label=r"$\alpha=0.1$")       # r in omschrijving label omwille van LaTeX-code
plt.plot(xwaarden, np.exp(-0.5 * xwaarden), label=r"$\alpha=0.5$")
plt.plot(xwaarden, np.exp(-1 * xwaarden), label=r"$\alpha=1$")
plt.plot(xwaarden, np.exp(-5 * xwaarden), label=r"$\alpha=5$")
plt.plot(xwaarden, np.exp(-10 * xwaarden), label=r"$\alpha=10$")
plt.xlabel(r"Afstand $d_{ij}$")                 
plt.ylabel(r"Kans op verbinding $v_{ij}$")
plt.legend(loc=0)

plt.show()

**Exercise 2**: Think carefully about the meaning of $\alpha$. What if $\alpha=0$? What if $\alpha$ is very large?

Answer:

In [ ]:
def sample_verbindingsmatrix(D, alpha=1.0):
    """Genereren van verbindingsmatrix afhankelijk van afstandsmatrix en alpha."""
   
    # verbindingsmatrix heeft dezelfde dimensie als afstandsmatrix, beide zijn vierkant
    n = D.shape[1]             # aantal kolommen in D is gelijk aan populatiegrootte
    
    # matrix aanmaken met 0 en 1 om verbindingen voor te stellen
    # alle elementen op diagonaal zijn nul, matrix is symmetrisch
    A = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
                 # kans op een verbinding
                 p = np.exp(- alpha * D[i,j])
                 # met een kans van p, maak een verbinding tussen i en j
                 if p > np.random.rand():
                        A[i,j] = 1
                        A[j,i] = 1      # symmetrische matrix
    return A


In [ ]:
# verbindingsmatrix van netwerk genereren voor alpha = 10
V = sample_verbindingsmatrix(D, alpha=10)
print(V)        # elke matrix kan gebruikt worden om figuur te representeren  
print(V.min(), V.max())

In [ ]:
# visualiseren dat V uit nullen en enen bestaat
plt.imshow(V, cmap="gray")   # elke matrix kan gebruikt worden als representatie voor afbeelding, 0 zwart, 1 wit 

#### Representing the network with a graph.

For this you write a new function in Python.<br> Infected individuals will be displayed in red, resistant in green and susceptible in yellow. So you will use a coloured chart. <br>If the state of the nodes has not been given, color them blue.
Thus, the list of points (nodes) of the network also corresponds to a list of states, where the first state corresponds to the first node, the second state with the second node, etc.

In [ ]:
 def plot_netwerk(X, V, toestanden=None):
    """Graaf van het netwerk.""" 
    n = V.shape[1]          # populatiegrootte is gelijk aan aantal kolommen van V
    
    # van elke knoop kleur nagaan en lijst van maken
    if toestanden is None:
        # geen toestanden gegeven, alle knopen zijn blauw
        knoop_kleuren = "blue"
    else:
        kleur_map = {"S" : "yellow", "I" : "red", "R" : "green"}    # dictionary
        knoop_kleuren = [kleur_map[toestand] for toestand in toestanden]
    
    
    plt.figure(figsize=(15,10))
    
    plt.axis("off")  # bij graaf geen assen  
    
    # plot n knopen, eerste kolom van X bevat x-coördinaat, tweede kolom y-coördinaat in juiste kleur
    plt.scatter(X[:,0], X[:,1], color=knoop_kleuren, zorder=1)    # zorder=1: punten op bovenste layer van graaf
    
    # teken verbindingen in grijs
    # n is populatiegrootte en V[i,j] is waarde van verbinding (0 of 1)
    # als V[i,j] = 1, dan lijnstuk tussen i-de en j-de knoop
    # plot om i-de en j-de knoop te verbinden
    # i-de en j-de knoop staan op i-de en j-de rij van X, dus X[i,j] nodig met x'n in eerste kolom daarvan en y's in tweede
    for i in range(n):
        for j in range(i+1, n):
            if V[i,j] == 1:
                plt.plot(X[[i,j],0], X[[i,j],1], alpha=0.8, color="grey", zorder=0)    # zorder=0: lijnen onderste layer van graaf
    plt.scatter([], [], color="yellow", label="S")       # lege punten om labels al te kunnen tonen
    plt.scatter([], [], color="red", label="I")
    plt.scatter([], [], color="green", label="R")
    plt.legend(loc=0)
    
    plt.show()

In [ ]:
plot_netwerk(X, V)       # knopen en verbindingen van ons netwerk plotten, nog zonder toestanden

#### Now assign an initial state to each of the nodes.

Initially, everyone is in state $S$, except for five random individuals who are infected.

In [ ]:
n_inf = 5  # initieel aantal geïnfecteerden

#lijst maken van initiële toestanden 
initiele_toestanden = ["S"] * n         # lijst maken van 200 S'n
initiele_toestanden[0: n_inf] = ["I"] * n_inf  # 5 S'n vervangen door I, maakt niet uit welke

In [ ]:
print(initiele_toestanden)
print(len(initiele_toestanden))

In [ ]:
plot_netwerk(X, V, initiele_toestanden)     # knopen en verbindingen van ons netwerk plotten, nu met initiële toestanden

#### Transition from one state to another

So you need a function that continuously converts the state at time $t$ to the state at time $t+1$. This is a fairly complicated function! The transition between times is called a *time step*.

In [ ]:
def update_toestand(toestanden, V, p_inf=1, p_res=0):
    "Functie die toestand aanpast naar nieuwe toestand per tijdstap."
    n = len(toestanden)        # aantal toestanden is populatiegrootte
    nieuwe_toestanden = []     # maak lijst om de nieuwe toestanden in op te slaan
    
    for i, toestand in enumerate(toestanden):         # ga lijst toestanden af en houd overeenkomstige index bij
        if toestand == "S":                           # persoon i is vatbaar
            # tel aantal geïnfecteerden die persoon i kent
            n_inf_kennissen = 0
            for j in range(n):
                if V[i,j] == 1 and toestanden[j] == "I":     # als persoon i in contact met geïnfecteerde persoon
                    n_inf_kennissen += 1
            # kans dat persoon i ziek wordt door een zieke kennis
            p_ziekte = 1 - (1 - p_inf)**n_inf_kennissen
            # effectief besmet of niet
            if (p_ziekte > np.random.rand()):
                toestand = "I" 
            else:
                toestand = "S"
            nieuwe_toestanden.append(toestand)
        elif toestand == "I":                          # persoon i is vatbaar
            # persoon die geïnfecteerd is, kan resistent worden
            # effectief besmet of niet
            if (p_res > np.random.rand()):
                toestand = "R"  
            else:
                toestand = "I"
            nieuwe_toestanden.append(toestand)
        elif toestand == "R":                          # persoon i is resistent
            # resistente personen blijven resistent
            nieuwe_toestanden.append("R")
    
    return nieuwe_toestanden

In [ ]:
# initiële toestanden updaten voor bepaalde p_inf en p_res voor één tijdstap
p_inf = 0.1
p_res = 0.01

nieuwe_toestanden = update_toestand(initiele_toestanden, V, p_inf, p_res)

print("aantal infecties op t = 0:", 5)
print("aantal infecties op t = 1:", nieuwe_toestanden.count("I"))

In [ ]:
plot_netwerk(X, V, nieuwe_toestanden)         # knopen en verbindingen van ons netwerk plotten, nu met toestanden op t = 1

#### Simulation evolution of states

You repeat this for a whole series of time steps using a for-loop:

In [ ]:
def simuleer_epidemie(init_toestanden, V, tijdstappen, p_inf=1, p_res=0):
    """Simulatie van evolutie toestanden."""
    # sla de toestanden op in een lijst van lijsten
    toestanden_lijst = [init_toestanden]     # lijst huidige toestanden wordt als eerste element in toestanden_lijst gestopt
    toestanden = init_toestanden
    for t in range(tijdstappen):
        toestanden = update_toestand(toestanden, V, p_inf, p_res)
        toestanden_lijst.append(toestanden)
    return toestanden_lijst

Do this once for 100 time steps.

In [ ]:
# simulatie van evolutie toestanden van initiële toestand over 100 tijdstappen
simulatie = simuleer_epidemie(initiele_toestanden, V, 100, p_inf, p_res)   # nog steeds p_inf = 0.1 en p_res = 0.01

View some snapshots through time now (at time steps 0, 10, 20, 50, 70 and 100).

In [ ]:
# verloop na 0, 10, 20, 50, 70 en 100 tijdstappen
for t in [0, 10, 20, 50, 70, 100]:
    toestanden = simulatie[t]             # simulatie is lijst van toestanden van toestanden
    print("tijdstip {}: {} geïnfecteerd, {} resistent".format(t, toestanden.count("I"), toestanden.count("R")))
    plot_netwerk(X, V, toestanden)
    

You can more easily monitor progress using a chart. See how the ratios between susceptibles, infected, and resistant change over time:

In [ ]:
def plot_progressiekrommen(toestanden_lijst):
    """Evolutie cijfers."""
    tijdstappen = len(toestanden_lijst)     # aantal elementen in toestanden_lijst is gelijk aan aantal tijdstappen
    # tel het aantal personen voor elke toestand per tijdstap
    S = [toestanden.count("S") for toestanden in toestanden_lijst]
    I = [toestanden.count("I") for toestanden in toestanden_lijst]
    R = [toestanden.count("R") for toestanden in toestanden_lijst]
    
    plt.figure()
    
    plt.plot(range(tijdstappen), I, color="purple", label="I")
    plt.plot(range(tijdstappen), S, color="orange", label="S")
    plt.plot(range(tijdstappen), R, color="green", label="R")
    plt.legend(loc=0)
    plt.xlabel("Tijd")
    plt.ylabel("Aantal personen")
    
    plt.show()

In [ ]:
def plot_progressievlakken(toestanden_lijst):
    """Evolutie cijfers."""
    tijdstappen = len(toestanden_lijst)     # aantal elementen in toestanden_lijst is gelijk aan aantal tijdstappen
    # tel het aantal personen voor elke toestand per tijdstap
    S = [toestanden.count("S") for toestanden in toestanden_lijst]
    I = [toestanden.count("I") for toestanden in toestanden_lijst]
    R = [toestanden.count("R") for toestanden in toestanden_lijst]
    
    plt.figure()
    
    plt.stackplot(range(tijdstappen), I, S, R,
                    labels=["I", "S", "R"], colors=["red", "yellow", "lightgreen"])
    plt.legend(loc=0)
    plt.xlabel("Tijd")
    plt.ylabel("Aantal personen")
    
    plt.show()

In [ ]:
plot_progressiekrommen(simulatie)

In [ ]:
plot_progressievlakken(simulatie)

**Exercise 3**: If too many people become sick too quickly, the health system can be overwhelmed, with catastrophic consequences! To avoid this, the principle of *social distancing* is applied: people must avoid social contact as much as possible. This ensures that the disease is passed on more slowly.- You can simulate social distancing by setting $\alpha$ higher, for example at 25. Do this. Do you see why the result is called the '*flatten the curve*'-effect?

<div class="alert alert-box alert-info">
Do you want to download this notebook, but has the file become too large due to the graphs?<br>Then first remove the output of the cells by choosing <b>Cell > All output > Clear</b> in the menu.You can also save the notebook as a PDF or print it out, just like you would do with a web page.</div>

<img src="images/cclic.png" alt="Banner" align="left" width="100"/><br><br>
This notebook by M. Stock and F. Wyffels for Dwengo vzw is licensed under a <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License</a>.